# VM0047 on a real plot: what has happened so far, and what to expect over ten years

This notebook applies Verra's VM0047 rules to a 1.27 ha plot in Chikkamagaluru, treated as a **new arecanut planting** made in June 2026, the planting season in which the proof of concept was built. It shows the plot, the record so far, and the credits expected over ten years.

**Read this first.**

- The plot is real. The planting is a what-if: the real plot is an established garden, which VM0047 would not credit as a new project.
- Tree sizes are measured values from a Malnad field trial (Bhoomika et al. 2023), not measurements of this plot. The years in between are modelled.
- No credits are issued here. Only a registry issues credits, after an accredited verifier has checked field measurements.

Sections 1 to 11 need no login. Section 12 is optional and needs Earth Engine.

In [ ]:
# Setup: get the repository if this notebook is running on its own (for example in Colab).
import os, sys
REPO_URL = "https://github.com/satvikkalyanpur/mrv-vm0047-exante.git"   # change if your repository has another name
if not os.path.exists("mrv"):
    if os.path.exists("../mrv"):
        os.chdir("..")
    else:
        !git clone -q {REPO_URL} repo
        os.chdir("repo")
sys.path.insert(0, os.getcwd())

import json
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt
from mrv import pipeline, figures, benchmark, ledger, observe
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 110)
plt.rcParams["figure.dpi"] = 110
result = pipeline.run()
sc, rules, s = result.scenario, result.rules, result.summary
print("Engine loaded.")

## 1. The plot

The same parcel as our proof of concept: four corners, 1.27 ha, near 13.400° N, 75.759° E.

In [ ]:
figures.parcel_map(sc["site"]["parcel_geojson"], sc["site"]["area_ha"]); plt.show()

The boundary on satellite imagery (loads in Colab or Jupyter; it does not show in GitHub's preview):

In [ ]:
try:
    import folium
    ring = json.load(open(sc["site"]["parcel_geojson"]))["features"][0]["geometry"]["coordinates"][0]
    lat, lon = observe.parcel_centroid(sc["site"]["parcel_geojson"])
    m = folium.Map(location=[lat, lon], zoom_start=17, tiles=None)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Esri World Imagery", name="Satellite").add_to(m)
    folium.Polygon([(p[1], p[0]) for p in ring], color="#D9722A", weight=3, fill=False).add_to(m)
    display(m)
except ImportError:
    print("folium is not installed; run `pip install folium` to see the satellite map.")

## 2. What has been observed at the plot

One satellite measurement of this plot exists already: the proof of concept measured a mean NDVI of 0.873 from Sentinel-2 for November 2024 to March 2025, which is dense, established canopy.

The cell below can add rainfall (NASA POWER) and greenness (MODIS, 250 m) from November 2024 to today. Both are free and need no login. Set `COLLECT = True` to fetch them; the files are saved in `data/observed/` so the record stays with the repository.

In [ ]:
COLLECT = False
if COLLECT:
    report = observe.collect(sc["site"]["parcel_geojson"], "data/observed",
                             date(2024, 11, 1), date.fromisoformat(sc["timeline"]["as_of"]))
    for k, v in report.items(): print(f"{k:26s} {v}")
obs = observe.load("data/observed")
print("Observed files present:", ", ".join(obs) or "none")
figures.observed_record(obs, sc["timeline"]["project_start"], sc["timeline"]["as_of"]); plt.tight_layout(); plt.show()
obs["ndvi_poc"]

## 3. The rules

Every number in this table is a rule of VM0047 or of the two Verra tools it points to. None is our choice.

In [ ]:
r = rules
pd.DataFrame([
    ["Carbon fraction of dry biomass", r["constants"]["carbon_fraction"], "VM0047 parameter CF (IPCC 2006)"],
    ["Carbon to CO2", "44/12", "VM0047 Eq. (1)"],
    ["Census-based approach limit", f'{r["approach_selection"]["census_max_planting_units_per_ha"]} planting units/ha', "VM0047 v1.1 applicability"],
    ["Uncertainty, ex ante", f'at least {r["uncertainty"]["exante_minimum_deduction"]:.0%}', "VM0047 Section 8.6"],
    ["Uncertainty, ex post", f'only the part above {r["uncertainty"]["allowance"]:.0%} (90% CI)', "VM0047 Eqs. (27), (28)"],
    ["Performance benchmark", "credit only growth beyond matched control plots", "VM0047 Appendix 1, Eqs. (A5), (A6)"],
    ["Ex-ante horizon", f'{r["exante"]["horizon_years"]} years', "VM0047 Section 8.6"],
    ["Monitoring interval", f'at most {r["monitoring"]["max_interval_years"]} years', "VM0047 Section 9"],
    ["Leakage defaults", f'{r["leakage"]["supply_increase_share"]:.0%} supply response, {r["leakage"]["new_land_share"]:.0%} from new land', "VMD0054"],
    ["Buffer", f'at least {r["buffer"]["minimum_rating"]}%, fail above {r["buffer"]["fail_overall"]}', "AFOLU Non-Permanence Risk Tool v4.2"],
], columns=["Rule", "Value", "Source"])

## 4. The planting, and which VM0047 approach applies

Sparse planting that leaves the land use unchanged is *census-based*. Dense planting that closes canopy over more than a hectare is *area-based*, and must pass a performance benchmark against control plots.

In [ ]:
est = sc["establishment"]
print("Scenario :", sc["name"])
print("Planted  :", sc["timeline"]["project_start"], "on", sc["land_at_start"]["state"])
print(f'Spacing  : {sc["stand"]["spacing_m"]} m x {sc["stand"]["spacing_m"]} m = {s["palms_per_ha"]:.0f} palms/ha, '
      f'{s["palms_per_ha"] * s["area_ha"]:.0f} palms on {s["area_ha"]} ha')
print(f'Seedlings: {est["seedling_age_at_planting_months"]} months old at planting; bearing starts after {est["pre_bearing_years"]} years (CPCRI)')
print("Approach :", result.approach.upper(), "-", result.approach_reason)

Quantifying carbon does not make a project eligible. A real project would still have to show these:

In [ ]:
pd.DataFrame([c.__dict__ for c in result.checks]).rename(columns={"rule": "Condition", "status": "Status", "note": "Note"})

## 5. Biomass at year 10

Height and diameter of 10-year-old palms are measured values for eight cultivars in Malnad. Biomass per palm comes from a species equation fitted to 40 palms that were cut down and weighed in India (Das et al. 2021, height model): 80% above ground, 20% below.

In [ ]:
result.cultivars.round(2)

In [ ]:
print(f'Spread between cultivars (90% CI half-width): {s["uncertainty_estimated"]:.1%}')
print(f'Uncertainty deduction applied               : {s["uncertainty_deducted"]:.0%}  (VM0047 ex-ante floor is 10%)')

## 6. So far: the first 5 months

Month by month from planting to today. Field stages are CPCRI's recommended practice. Stock and credits are modelled: palms this young are far below the height range of the biomass equation, so treat these as indicative. Nothing is issued before the first verification.

In [ ]:
progress = pipeline.progress_to_date(result)
figures.progress_so_far(progress, s["as_of"]); plt.tight_layout(); plt.show()
show = progress[progress.month.isin([0, 6, 12, progress.month.max()])]
show[["month", "date", "mean_height_m", "stock_change_tco2e", "credits_accrued", "credits_issued", "field_stage"]].round(2)

## 7. Expected: ten years

Credits accrue every year but are issued only at verification, here in years 5 and 10.

In [ ]:
figures.credits_over_time(result); plt.tight_layout(); plt.show()
cols = ["year", "calendar_year", "mean_height_m", "stock_change_tco2e", "net_removals_cumulative_tco2e",
        "buffer_in_year_tco2e", "credits_accrued_cumulative", "credits_issued_cumulative"]
result.schedule[cols].round(2)

## 8. From carbon stored to credits issuable

The reference case sets every deduction at the lowest value the methodology allows, so it is an upper bound.

In [ ]:
figures.waterfall(result); plt.tight_layout(); plt.show()
print(f'Carbon stored by year 10 : {s["stock_change_tco2e"]:.1f} t CO2e')
print(f'Credits issuable         : {s["credits_issuable"]:.1f}  = {s["credits_per_ha_per_year"]:.2f} per hectare per year')
illustrative = pipeline.run(pipeline.with_overrides(sc, **sc["illustrative_case"]), rules)
print(f'With illustrative deductions (benchmark 13%, uncertainty 15%, buffer 20%): {illustrative.summary["credits_issuable"]:.1f}')

## 9. Reality check against measured stands

Is the year-10 figure believable? Das et al. measured whole stands of 10 to 35 years in Assam. Their palms hold 10.5 to 12.0 kg of carbon each. The model gives less than that per palm at year 10, so it errs on the low side.

In [ ]:
check = pipeline.reality_check(result)
figures.reality_check(result, check); plt.tight_layout(); plt.show()
check[["source", "location", "age_years", "palms_per_ha", "carbon_t_per_ha", "carbon_kg_per_palm", "pools"]].round(2)

## 10. What moves the answer, and where the trees go

One input is changed at a time. The biomass equation matters far more than any deduction.

In [ ]:
sens = pipeline.sensitivity(sc, rules)
figures.sensitivity_ranges(sens, s["credits_issuable"]); plt.tight_layout(); plt.show()
crop = pipeline.cropland_alternative(result)
print("If the same palms replaced a working field instead of fallow (VMD0054):")
print(f'  new land brought into production elsewhere: {crop["new_land_ha"]:.2f} ha')
print(f'  leakage cancels all {crop["removals_before_leakage_tco2e"]:.0f} t CO2e of removals if that land holds more than '
      f'{crop["breakeven_carbon_stock_tc_per_ha"]:.0f} t C per ha')

Why the real, existing garden earns nothing: for dense planting, growth is compared with matched control plots and only the difference is credited. The slopes below are illustrative, not satellite data; they show how the rule behaves.

In [ ]:
T = benchmark.Trend
cases = {
    "New planting on fallow, controls flat":      (T(0.150, 0.010, 0.0001, 10), T(0.002, 0.004, 0.60, 10)),
    "New planting, controls regrowing slowly":    (T(0.150, 0.010, 0.0001, 10), T(0.020, 0.005, 0.01, 10)),
    "Existing garden vs neighbouring gardens":    (T(0.010, 0.004, 0.03, 10),   T(0.009, 0.004, 0.04, 10)),
}
pd.DataFrame([{"Case": k, "Z": round(benchmark.z_statistic(p, c), 2),
               "Benchmark PB": round(benchmark.performance_benchmark(p, c), 2),
               "Share credited": f"{1 - benchmark.performance_benchmark(p, c):.0%}"} for k, (p, c) in cases.items()])

## 11. Integrity log

Each input file and each headline result is recorded in a SHA-256 hash chain. Changing any entry breaks every hash after it.

In [ ]:
import copy
chain = pipeline.build_ledger(result, pipeline.INPUT_FILES)
for e in chain:
    print(f'{e.get("file", e.get("name")):45s} {str(e.get("value", "")):>10s}  {e["hash"][:16]}')
print("\nChain verifies:", ledger.verify_chain(chain))
tampered = copy.deepcopy(chain); tampered[-1]["value"] = 99.0
print("After editing the credit figure:", ledger.verify_chain(tampered))

## 12. Optional: the plot at 10 m from Sentinel-2 (needs Earth Engine)

Monthly NDVI of the parcel itself, from November 2024 to today, saved to `data/observed/ndvi_sentinel2_10m.csv`. Cloudy monsoon months come back empty. Set `RUN_EE = True` and enter your Earth Engine project ID.

In [ ]:
RUN_EE = False
EE_PROJECT = "your-earth-engine-project-id"
if RUN_EE:
    !pip install -q earthengine-api
    import ee
    ee.Authenticate(); ee.Initialize(project=EE_PROJECT)
    ring = json.load(open(sc["site"]["parcel_geojson"]))["features"][0]["geometry"]["coordinates"]
    aoi = ee.Geometry.Polygon(ring)
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(aoi)
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30)))
    rows, end = [], date.fromisoformat(sc["timeline"]["as_of"])
    y, mo = 2024, 11
    while (y, mo) <= (end.year, end.month):
        start = ee.Date.fromYMD(y, mo, 1)
        col = s2.filterDate(start, start.advance(1, "month"))
        n = col.size().getInfo()
        ndvi = None
        if n > 0:
            img = col.median().normalizedDifference(["B8", "B4"]).rename("NDVI")
            ndvi = img.reduceRegion(ee.Reducer.mean(), aoi, 10).get("NDVI").getInfo()
        rows.append({"date": f"{y}-{mo:02d}-15", "ndvi": ndvi, "images": n})
        y, mo = (y + 1, 1) if mo == 12 else (y, mo + 1)
    s2_series = pd.DataFrame(rows)
    s2_series.to_csv("data/observed/ndvi_sentinel2_10m.csv", index=False)
    figures.observed_record(observe.load("data/observed"), sc["timeline"]["project_start"], sc["timeline"]["as_of"]); plt.show()
    display(s2_series)
else:
    print("Skipped. Everything above runs without Earth Engine.")

## What is measured, cited and assumed

| Item | Status |
|---|---|
| Plot boundary and area | Real: the proof-of-concept parcel |
| NDVI 0.873, Nov 2024 to Mar 2025 | Measured on this plot (Sentinel-2, proof of concept) |
| VM0047, VMD0054 and buffer rules | Cited from Verra documents |
| Palm height and diameter at age 10 | Cited: measured in Malnad (Bhoomika et al. 2023) |
| Biomass equation and root share | Cited: Das et al. 2021, India |
| Spacing, planting season, seedling age, field stages | Cited: CPCRI package of practices |
| Planting on fallow land in June 2026 | Assumed: this is what makes it a what-if |
| Growth between planting and year 10 | Assumed: linear height growth |
| Palm survival of 100% | Assumed: no mortality data |
| Performance benchmark of 0 | Assumed: needs satellite control plots |
| Buffer rating of 12 | Floor: a real rating needs the full risk tool |